In [3]:
import pandas as pd

In [3]:
!pip install requests beautifulsoup4 pandas

In [5]:
import requests
from bs4 import BeautifulSoup
import time

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:
BASE_URL = "https://www.shl.com"
CATALOG_URL = "https://www.shl.com/products/product-catalog/"

def get_page(url):
    """Fetch a page and return BeautifulSoup object"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    response = requests.get(url, headers=headers)
    return BeautifulSoup(response.text, "html.parser")

def scrape_catalog_page(start):
    """
    Scrape one page of Individual Test Solutions
    type=1 means Individual Test Solutions (we ignore type=2 which is Pre-packaged)
    start = page offset (0, 12, 24, ...)
    """
    url = f"{CATALOG_URL}?start={start}&type=1"
    soup = get_page(url)
    
    assessments = []
    
    tables = soup.find_all("table")
    
    for table in tables:
        header = table.find("th")
        if header and "Individual Test Solutions" in header.text:
            rows = table.find_all("tr")[1:] 
            for row in rows:
                cols = row.find_all("td")
                if len(cols) >= 4:
                    name_tag = cols[0].find("a")
                    if name_tag:
                        name = name_tag.text.strip()
                        relative_url = name_tag.get("href", "")
                        full_url = BASE_URL + relative_url
                    else:
                        continue
                    remote = "Yes" if cols[1].find("span") else "No"
                                       
                    adaptive = "Yes" if cols[2].find("span") else "No"
                    
                    test_type = cols[3].text.strip()
                    
                    assessments.append({"name": name,"url": full_url,"remote_testing": remote,"adaptive_irt": adaptive,"test_type": test_type})

    return assessments

print("Functions defined successfully!")

Functions defined successfully!


In [11]:
all_assessments = []

for start in range(0, 384, 12):  
    print(f"Scraping page with start={start}...")
    page_data = scrape_catalog_page(start)
    
    if not page_data:
        print(f"  No data found at start={start}, stopping.")
        break
    
    all_assessments.extend(page_data)
    print(f"  Found {len(page_data)} assessments. Total so far: {len(all_assessments)}")
    
    time.sleep(1)

print(f"\n✅ Scraping complete! Total assessments collected: {len(all_assessments)}")

Scraping page with start=0...
  Found 12 assessments. Total so far: 12
Scraping page with start=12...
  Found 12 assessments. Total so far: 24
Scraping page with start=24...
  Found 12 assessments. Total so far: 36
Scraping page with start=36...
  Found 12 assessments. Total so far: 48
Scraping page with start=48...
  Found 12 assessments. Total so far: 60
Scraping page with start=60...
  Found 12 assessments. Total so far: 72
Scraping page with start=72...
  Found 12 assessments. Total so far: 84
Scraping page with start=84...
  Found 12 assessments. Total so far: 96
Scraping page with start=96...
  Found 12 assessments. Total so far: 108
Scraping page with start=108...
  Found 12 assessments. Total so far: 120
Scraping page with start=120...
  Found 12 assessments. Total so far: 132
Scraping page with start=132...
  Found 12 assessments. Total so far: 144
Scraping page with start=144...
  Found 12 assessments. Total so far: 156
Scraping page with start=156...
  Found 12 assessments. 

In [13]:
df = pd.DataFrame(all_assessments)

df = df.drop_duplicates(subset=["url"])

print(f"Total unique assessments: {len(df)}")
print(f"\nFirst 5 rows:")
df.head()

Total unique assessments: 377

First 5 rows:


,name,url,remote_testing,adaptive_irt,test_type
0,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,Yes,No,A\nE\nB\nC\nD\nP
1,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,Yes,Yes,K
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,Yes,No,K
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,Yes,No,K
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,Yes,No,K


In [15]:
def scrape_assessment_detail(url):
    """
    Visit each assessment page and get its description
    """
    try:
        soup = get_page(url)
        
        description = ""
        
        desc_div = soup.find("div", class_="product-catalogue-training-calendar__row")
        if desc_div:
            description = desc_div.text.strip()
        
        if not description:
            meta = soup.find("meta", {"name": "description"})
            if meta:
                description = meta.get("content", "")
        
        return description
    except Exception as e:
        return ""

descriptions = []

for i, row in df.iterrows():
    desc = scrape_assessment_detail(row["url"])
    descriptions.append(desc)
    
    if (i + 1) % 20 == 0:
        print(f"Progress: {i+1}/{len(df)} assessments processed...")
    
    time.sleep(2)  

df["description"] = descriptions
print("✅ Descriptions scraped!")

Progress: 20/377 assessments processed...
Progress: 40/377 assessments processed...
Progress: 60/377 assessments processed...
Progress: 80/377 assessments processed...
Progress: 100/377 assessments processed...
Progress: 120/377 assessments processed...
Progress: 140/377 assessments processed...
Progress: 160/377 assessments processed...
Progress: 180/377 assessments processed...
Progress: 200/377 assessments processed...
Progress: 220/377 assessments processed...
Progress: 240/377 assessments processed...
Progress: 260/377 assessments processed...
Progress: 280/377 assessments processed...
Progress: 300/377 assessments processed...
Progress: 320/377 assessments processed...
Progress: 340/377 assessments processed...
Progress: 360/377 assessments processed...
✅ Descriptions scraped!


In [17]:
df.to_csv("shl_assessments.csv", index=False)

print(f"✅ Saved to shl_assessments.csv")
print(f"Total assessments: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample data:")
df.head(3)

✅ Saved to shl_assessments.csv
Total assessments: 377

Columns: ['name', 'url', 'remote_testing', 'adaptive_irt', 'test_type', 'description']

Sample data:


,name,url,remote_testing,adaptive_irt,test_type,description
0,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,Yes,No,A\nE\nB\nC\nD\nP,Description\nThis report is designed to be giv...
1,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,Yes,Yes,K,Description\nThe.NET Framework 4.5 test measur...
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,Yes,No,K,Description\nMulti-choice test that measures t...


In [19]:
print("=== DATA VALIDATION ===")
print(f"Total assessments: {len(df)}")
print(f"Minimum required: 377")
print(f"Requirement met: {len(df) >= 377} ✅" if len(df) >= 377 else f"Requirement NOT met ❌")
print(f"\nTest Types found: {df['test_type'].unique()}")
print(f"\nRemote Testing counts:\n{df['remote_testing'].value_counts()}")
print(f"\nAdaptive IRT counts:\n{df['adaptive_irt'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

=== DATA VALIDATION ===
Total assessments: 377
Minimum required: 377
Requirement met: True ✅

Test Types found: ['A\nE\nB\nC\nD\nP' 'K' 'S' 'P' 'E' 'S\nK' 'K\nS' 'B\nS' 'B\nP\nS' 'C\nP'
 'P\nC' 'B' 'C\nK' 'C' 'A\nP' 'D' 'A\nK\nS' 'P\nS' 'C\nA\nP' 'A\nC\nP' 'A'
 'B\nK\nS\nA' 'S\nB' 'P\nP' 'A\nS' 'A\nK' 'D\nP']

Remote Testing counts:
remote_testing
Yes    377
Name: count, dtype: int64

Adaptive IRT counts:
adaptive_irt
No     340
Yes     37
Name: count, dtype: int64

Missing values:
name              0
url               0
remote_testing    0
adaptive_irt      0
test_type         0
description       0
dtype: int64
